# Applicator

I want to apply a theory to a response. In principle, the response should just be a sentence, but it needs to have been read and SpaCy'd first. So for the moment, I'll just assume that we're applying to sentences from the data file. Which, fortunately, is all encoded in the theory object, having been extracted from `bea_3way_data_pickle`.

Want two outputs: one of the results, one of the applications.

The theory itself will be the output from `bea_3way_theory_generator.ipynb/induce_theory`.

Want `apply_theory(theory, original_response_id)`.

Actually, a better plan might be to just have a function which applies the theory to the embedded trial set.

`apply_trial_set(theory)`

In [1]:
import amati_bea as amati

In [2]:
import pandas as pd

from sklearn.metrics import f1_score, cohen_kappa_score

In [3]:
from operator import itemgetter
import pickle
import datetime
import re
import glob

## Get the data

We'll just be able to pull out the relevant data from the saved theory.

In [4]:
with open(
    "/Users/alistair.willis/repos/bea-2026/3-way/theories/run_20260303/theory_64393e93-5585-42f9-ad22-ffa71219573b_2026-03-04_02:35:57.762552.pickle", "rb"
) as fIn:
    theory = pickle.load(fIn)

In [5]:
theory.keys()

dict_keys(['theory', 'original_question_id', 'question_id', 'all_questions_df', 'all_responses_df', 'all_text_df', 'rules'])

Let's find a trial response. First, need to pull out the question for that theory:

In [6]:
theory["question_id"]

'q_18'

And then get the associated responses:

In [7]:
theory["all_responses_df"].query("`question_id`==@theory['question_id']").query(
    "`partition`=='trial'"
).head()

,original_response_id,response_id,question_id,response,score,partition
2102,07cf8c34-4a35-4d99-8ad4-8e1358fbd089,r_0379,q_18,Die Bewegungsenergie des Windes wird zunächst ...,correct,trial
2103,faacc838-2de9-40d2-99d0-dec674a1ef50,r_1842,q_18,-Überflüssiger Strom wird effizient genutzt -n...,incorrect,trial
2104,f615be76-98ba-4c39-91f2-0b4c5354759a,r_2248,q_18,Die Bewegungsenergie fließt in eine erneuerbar...,correct,trial
2105,d700fe2d-d7e1-4792-abe0-b193289e8c6e,r_2567,q_18,die WIndenergie wird in Gas um gewandelt\n,partially_correct,trial
2106,a2857c1f-8410-4c22-9876-c64b58ccae0d,r_2818,q_18,Energie wird in einem Elektrolyt Kraftwerk zu ...,partially_correct,trial


So let's just bag the first of those:

In [8]:
TEST_RESPONSE_ID = (
    theory["all_responses_df"]
    .query("`question_id`==@theory['question_id']")
    .query("`partition`=='trial'")
    .iloc[0, 0]
)

TEST_RESPONSE_ID

'07cf8c34-4a35-4d99-8ad4-8e1358fbd089'

Reverse ferret, let's apply to the trial set:

In [9]:
def apply_trial_set(theory):

    question_id=theory["question_id"]
    original_question_id=theory["original_question_id"]

    question_ss = theory['all_questions_df'].set_index("original_question_id").T[
        original_question_id
    ]
    
    trial_responses_df = theory['all_responses_df'].query("`question_id`==@question_id").query(
        "`partition`=='trial'"
    )

    trial_texts_used = [
        question_id,
        question_ss["correct_id"],
        question_ss["partially_correct_id"],
        question_ss["incorrect_id"],
    ] + list(trial_responses_df["response_id"])

    trial_text_df = theory['all_text_df'].query("`id`.isin(@trial_texts_used)")[
        ["id", "i", "lemma"]
    ]

    results_df = amati.evaluate_theory(
        theory['theory'],
        responses_df=trial_responses_df,
        text_df=trial_text_df,
        rulesstring=theory['rules'],
        question_ss=question_ss,
    )

    compare_df = (
        trial_responses_df[["response_id", "score"]]
        .rename({"score": "actual"}, axis="columns")
        .set_index("response_id")
        .assign(
            prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
        )
    )

    return {"theory": theory, "results": results_df, "compare": compare_df}

In [10]:
def get_rule_applications(theory):

    question_id=theory["question_id"]
    original_question_id=theory["original_question_id"]

    question_ss = theory['all_questions_df'].set_index("original_question_id").T[
        original_question_id
    ]
    
    trial_responses_df = theory['all_responses_df'].query("`question_id`==@question_id").query(
        "`partition`=='trial'"
    )

    trial_texts_used = [
        question_id,
        question_ss["correct_id"],
        question_ss["partially_correct_id"],
        question_ss["incorrect_id"],
    ] + list(trial_responses_df["response_id"])

    trial_text_df = theory['all_text_df'].query("`id`.isin(@trial_texts_used)")[
        ["id", "i", "lemma"]
    ]

    results_dict = amati.apply_theory(
        theory['theory'],
        responses_df=trial_responses_df,
        text_df=trial_text_df,
        rulesstring=theory['rules'],
        question_ss=question_ss,
    )

    """ compare_df = (
        trial_responses_df[["response_id", "score"]]
        .rename({"score": "actual"}, axis="columns")
        .set_index("response_id")
        .assign(
            prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
        )
    ) """

    return {"theory": theory, "applications": results_dict}

In [11]:
question_id = theory["question_id"]
original_question_id = theory["original_question_id"]

question_ss = (
    theory["all_questions_df"].set_index("original_question_id").T[original_question_id]
)

trial_responses_df = (
    theory["all_responses_df"]
    .query("`question_id`==@question_id")
    .query("`partition`=='trial'")
)

trial_texts_used = [
    question_id,
    question_ss["correct_id"],
    question_ss["partially_correct_id"],
    question_ss["incorrect_id"],
] + list(trial_responses_df["response_id"])

trial_text_df = theory["all_text_df"].query("`id`.isin(@trial_texts_used)")[
    ["id", "i", "lemma"]
]

In [13]:

amati.apply_theory(
    theory["theory"],
    responses_df=trial_responses_df,
    rulesstring=theory["rules"],
    question_ss=question_ss,
    text_df=trial_text_df,
)

{'r_0379': [{'result': 'SATISFIABLE',
   'selected_rule': ('rule1', 'selected_rule(rule1)'),
   'predicted_grade': ('correct', 'predicted_grade(correct)'),
   'parameters': [(1, '"umwandeln"', 'parameter(1,"umwandeln")'),
    (2, '"energie"', 'parameter(2,"energie")')],
   'positive_covered': ['r_0218',
    'r_0368',
    'r_0418',
    'r_0583',
    'r_0652',
    'r_0662',
    'r_0668',
    'r_1000',
    'r_1021',
    'r_1115',
    'r_1233',
    'r_1433',
    'r_1660',
    'r_1714',
    'r_1793',
    'r_1831',
    'r_1875',
    'r_2079',
    'r_2112',
    'r_2217',
    'r_2265',
    'r_2295',
    'r_2304',
    'r_2347',
    'r_2401',
    'r_2432',
    'r_2445',
    'r_2539',
    'r_2926',
    'r_3002',
    'r_3546',
    'r_3553',
    'r_3653',
    'r_3660',
    'r_3796',
    'r_3801',
    'r_4084',
    'r_4120',
    'r_4147',
    'r_4313',
    'r_4358',
    'r_4534',
    'r_4541',
    'r_4587',
    'r_4801',
    'r_4908',
    'r_5218',
    'r_5228',
    'r_5692',
    'r_5917',
    'r_59

In [14]:
o=apply_trial_set(theory)

In [15]:
o['results']

,0,1,2,3,4,5,6,7
response_id,,,,,,,,
r_0379,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2248,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2818,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3719,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_4642,correct,NaN,incorrect,NaN,NaN,NaN,NaN,NaN
r_4743,correct,NaN,NaN,correct,NaN,NaN,NaN,NaN
r_4750,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
t=get_rule_applications(theory)
t['applications']

{'r_0379': [{'result': 'SATISFIABLE',
   'selected_rule': ('rule1', 'selected_rule(rule1)'),
   'predicted_grade': ('correct', 'predicted_grade(correct)'),
   'parameters': [(1, '"umwandeln"', 'parameter(1,"umwandeln")'),
    (2, '"energie"', 'parameter(2,"energie")')],
   'positive_covered': ['r_0218',
    'r_0368',
    'r_0418',
    'r_0583',
    'r_0652',
    'r_0662',
    'r_0668',
    'r_1000',
    'r_1021',
    'r_1115',
    'r_1233',
    'r_1433',
    'r_1660',
    'r_1714',
    'r_1793',
    'r_1831',
    'r_1875',
    'r_2079',
    'r_2112',
    'r_2217',
    'r_2265',
    'r_2295',
    'r_2304',
    'r_2347',
    'r_2401',
    'r_2432',
    'r_2445',
    'r_2539',
    'r_2926',
    'r_3002',
    'r_3546',
    'r_3553',
    'r_3653',
    'r_3660',
    'r_3796',
    'r_3801',
    'r_4084',
    'r_4120',
    'r_4147',
    'r_4313',
    'r_4358',
    'r_4534',
    'r_4541',
    'r_4587',
    'r_4801',
    'r_4908',
    'r_5218',
    'r_5228',
    'r_5692',
    'r_5917',
    'r_59

In [18]:
t['applications']['r_1842']


[]

## Apply to all theories

In [ ]:
outputs_ls=[]

for f in glob.glob("theories/run_20260303/*"):
    with open(f, 'rb') as fIn:
        theory=pickle.load(fIn)
    outputs_ls.append(apply_trial_set(theory))

And now we should be able to get an overall assessment...

In [ ]:
outputs_ls[1]['results']

In [ ]:
outputs_ls[2]['results']

First, get a big table of all the predictions:

In [ ]:
predictions_df=pd.concat([o['results'] for o in outputs_ls])
predictions_df

In [ ]:
# Let's define an ordered category for the prediction:

prediction_type = pd.CategoricalDtype(
    categories=['incorrect', 'partially_correct', 'correct'],
    ordered=True
)
prediction_type

And we can use this type on the predictions DataFrame:

In [ ]:
predictions_df=predictions_df.astype(prediction_type)


Now, an interesting comparison might be whether we choose the first prediction or the last prediction. ie. Do later predictions override the earlier ones, or should the first prediction stand?

In [ ]:
predict_first_ss=predictions_df.bfill(axis='columns').iloc[:, 0].fillna('incorrect')
predict_first_ss

In [ ]:
predict_last_ss=predictions_df.ffill(axis='columns').iloc[:, -1].fillna('incorrect')
predict_last_ss

And now we need to do a comparison with the ground truth results.

In [ ]:
results_df = (
    theory["all_responses_df"]
    .query("`partition`=='trial'")
    .set_index("response_id")[["score"]]
    .rename({"score": "actual"}, axis="columns")
    .astype(prediction_type)
    .assign(predict_first=predict_first_ss, predict_last=predict_last_ss)
)

results_df

In [ ]:
results_df.query('`predict_first`!=`predict_last`')

In [ ]:
f1_score(results_df['actual'], results_df['predict_first'], average='weighted')

In [ ]:
f1_score(results_df['actual'], results_df['predict_last'], average='weighted')

OK, predict first seems slightly better, as expected, although they're both pretty dire. Now let's do the evaluation with CWK:

In [ ]:
cohen_kappa_score(results_df['actual'], results_df['predict_first'], weights='quadratic')

Ugghh... Although I wonder what the f1 score is:

In [ ]:
f1_score(results_df['actual'], results_df['predict_first'], average='weighted')

That's hardly going to set the world alight...